In [12]:
%pip install openpyxl

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
"""
Excel 空白单元格线性插值工具
功能：对 Excel 文件中所有工作表的空白单元格，按列方向进行线性插值填充
插值方法：利用空白单元格两侧最近的非空数值进行线性插值
"""

import openpyxl
import os
import argparse


def interpolate_cell(value1, value2, pos, total_gap):
    """
    在两个已知值之间进行线性插值
    value1: 左侧/上方已知值
    value2: 右侧/下方已知值
    pos: 当前空白单元格在间隙中的位置（从1开始）
    total_gap: 空白单元格总数
    """
    if value1 is None or value2 is None:
        return None
    try:
        v1 = float(value1)
        v2 = float(value2)
    except (ValueError, TypeError):
        return None
    # 线性插值公式: y = y1 + (y2 - y1) * (pos / (total_gap + 1))
    return v1 + (v2 - v1) * pos / (total_gap + 1)


def interpolate_row(row_values):
    """
    对一行数据进行线性插值，返回插值后的列表
    row_values: 某一行（或列）的值列表
    """
    result = list(row_values)
    n = len(result)

    # 找到所有非空的数值位置
    known_positions = []
    for i, val in enumerate(result):
        if val is not None and val != '':
            try:
                float(val)
                known_positions.append(i)
            except (ValueError, TypeError):
                pass

    if len(known_positions) < 2:
        # 已知值不足2个，无法插值
        return result

    # 遍历每对已知值之间的间隙进行插值
    for idx in range(len(known_positions) - 1):
        start = known_positions[idx]
        end = known_positions[idx + 1]
        gap = end - start - 1
        if gap <= 0:
            continue
        v1 = float(result[start])
        v2 = float(result[end])
        for i in range(1, gap + 1):
            interpolated = v1 + (v2 - v1) * i / (gap + 1)
            result[start + i] = round(interpolated, 6)

    return result


def process_worksheet(ws, direction='both'):
    """
    处理单个工作表，对空白单元格进行线性插值
    direction: 'row' 按行插值, 'col' 按列插值, 'both' 先按列再按行
    """
    max_row = ws.max_row
    max_col = ws.max_column

    # 收集所有数据
    data = []
    for row in ws.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col):
        row_data = []
        for cell in row:
            row_data.append(cell.value)
        data.append(row_data)

    modified_cells = 0

    if direction in ('col', 'both'):
        # 按列插值（每列从上到下）
        for col in range(max_col):
            col_values = [data[row][col] for row in range(max_row)]
            interpolated = interpolate_row(col_values)
            for row in range(max_row):
                if data[row][col] is None or data[row][col] == '':
                    if interpolated[row] is not None:
                        data[row][col] = interpolated[row]
                        modified_cells += 1

    if direction in ('row', 'both'):
        # 按行插值（每行从左到右）
        for row in range(max_row):
            row_values = list(data[row])
            interpolated = interpolate_row(row_values)
            for col in range(max_col):
                if (data[row][col] is None or data[row][col] == '') and interpolated[col] is not None:
                    data[row][col] = interpolated[col]
                    modified_cells += 1

    # 写回工作表
    for row in range(max_row):
        for col in range(max_col):
            cell = ws.cell(row=row + 1, column=col + 1)
            if data[row][col] is not None:
                cell.value = data[row][col]

    return modified_cells


def interpolate_excel(input_file, output_file=None, direction='both', sheet_names=None):
    """
    主函数：对 Excel 文件进行线性插值

    参数:
        input_file: 输入 Excel 文件路径
        output_file: 输出 Excel 文件路径（默认为原文件名加 _interpolated 后缀）
        direction: 插值方向 'row'/'col'/'both'
        sheet_names: 要处理的工作表名称列表（None 表示处理所有工作表）

    返回:
        总修改单元格数
    """
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"文件不存在: {input_file}")

    if output_file is None:
        base, ext = os.path.splitext(input_file)
        output_file = f"{base}_interpolated{ext}"

    wb = openpyxl.load_workbook(input_file)

    if sheet_names is None:
        sheet_names = wb.sheetnames

    total_modified = 0
    report = {}

    for name in sheet_names:
        if name not in wb.sheetnames:
            print(f"  警告: 工作表 '{name}' 不存在，已跳过")
            continue
        ws = wb[name]
        print(f"  正在处理工作表: {name} (行: {ws.max_row}, 列: {ws.max_column})")
        count = process_worksheet(ws, direction=direction)
        report[name] = count
        total_modified += count
        print(f"    ✓ 插值填充了 {count} 个单元格")

    wb.save(output_file)
    print(f"\n✅ 处理完成！共填充 {total_modified} 个空白单元格")
    print(f"📁 输出文件: {output_file}")

    return total_modified, output_file, report


# ==================== 演示 ====================
def demo_mode():
    """创建示例数据并演示插值效果"""
    demo_file = '/data/workspace/demo_input.xlsx'
    wb = openpyxl.Workbook()

    # Sheet1: 模拟温度变化数据（有缺失）
    ws1 = wb.active
    ws1.title = "温度数据"
    ws1['A1'] = '时间'
    ws1['B1'] = '温度(°C)'
    time_labels = ['6:00', '7:00', '8:00', '9:00', '10:00',
                   '11:00', '12:00', '13:00', '14:00', '15:00']
    temps = [18.5, 20.1, None, 23.8, None, 27.2, None, 30.5, None, 28.3]
    for i, (t, temp) in enumerate(zip(time_labels, temps), start=2):
        ws1[f'A{i}'] = t
        if temp is not None:
            ws1[f'B{i}'] = temp

    # Sheet2: 模拟二维表格数据（行列都有缺失）
    ws2 = wb.create_sheet("销售数据")
    headers = ['产品', 'Q1', 'Q2', 'Q3', 'Q4']
    ws2.append(headers)
    data = [
        ['A产品', 100, None, 150, None],
        ['B产品', None, 200, None, 300],
        ['C产品', 80, None, None, 160],
    ]
    for row in data:
        ws2.append(row)

    wb.save(demo_file)
    print(f"📝 已创建示例文件: {demo_file}\n")

    # 执行插值
    print("🔄 开始线性插值处理...\n")
    output_file = '/data/workspace/demo_output.xlsx'
    total, output_file, report = interpolate_excel(
        input_file=demo_file,
        output_file=output_file,
        direction='both'
    )

    # 展示结果
    print("\n" + "=" * 50)
    print("📊 插值结果展示")
    print("=" * 50)

    wb_out = openpyxl.load_workbook(output_file)
    for name in wb_out.sheetnames:
        ws = wb_out[name]
        print(f"\n【{name}】")
        for row in ws.iter_rows(min_row=1, max_row=ws.max_row,
                                min_col=1, max_col=ws.max_column, values_only=False):
            values = []
            for cell in row:
                if cell.value is None:
                    values.append(f"{cell.coordinate}: ⚠空")
                else:
                    if isinstance(cell.value, float):
                        values.append(f"{cell.coordinate}: {cell.value:.2f}")
                    else:
                        values.append(f"{cell.coordinate}: {cell.value}")
            print("  ".join(values))


In [14]:

total, output_file, report = interpolate_excel(
    input_file=r'C:\Users\zheng\Desktop\全50版本daq_export_20260804_101322.xlsx',
    output_file=r'C:\Users\zheng\Desktop\全50版本daq_export_20260804_101322_filled.xlsx',
    direction='both',          # 'row' / 'col' / 'both'
    sheet_names=None           # None = 所有工作表
)

  正在处理工作表: daq_export_20260804_101322 (行: 11841, 列: 20)
    ✓ 插值填充了 2984 个单元格
  正在处理工作表: Sheet1 (行: 990, 列: 13)
    ✓ 插值填充了 988 个单元格

✅ 处理完成！共填充 3972 个空白单元格
📁 输出文件: C:\Users\zheng\Desktop\全50版本daq_export_20260804_101322_filled.xlsx


In [15]:
if os.path.exists(output_file):
    os.startfile(output_file)